In [1]:
import matplotlib.pyplot as plt

from keras.layers.advanced_activations import LeakyReLU
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from tensorflow.keras import datasets, layers, models

from keras.optimizers import SGD
import h5py as h5
import os

import scipy.io
from scipy.io import loadmat
import cv2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn import metrics
import numpy as np




Using TensorFlow backend.


In [2]:
tf.__version__

'1.14.0'

In [9]:
from google.colab import drive
drive.mount('/content/gdrive',force_remount=True)

Mounted at /content/gdrive


In [10]:
cd /content/gdrive/My\ Drive/

/content/gdrive/My Drive


In [7]:
print(os.getcwd())

/content


In [0]:
#from google.colab import files
#uploaded = files.upload()

Data fetching and understand the train/val/test splits.  

In [0]:
# Open the file as readonly
h5f = h5.File('SVHN_single_grey1.h5', 'r')

# Load the training, test and validation set
X_train = h5f['X_train'][:]
y_train = h5f['y_train'][:]
X_test = h5f['X_test'][:]
y_test = h5f['y_test'][:]
X_val = h5f['X_val'][:]
y_val = h5f['y_val'][:]

In [12]:
X_train.shape

(42000, 32, 32)

In [13]:
X_test.shape

(18000, 32, 32)

In [14]:
y_train.shape

(42000,)

In [15]:
X_train.size

43008000

In [0]:
X_train_knn = np.reshape(X_train, (-1,1024))
X_test_knn = np.reshape(X_test, (-1,1024 ))

In [17]:
X_train_knn.shape

(42000, 1024)

Implement and apply an optimal k-Nearest Neighbor (kNN) classifier 

In [0]:
KNN = KNeighborsClassifier(n_neighbors= 9)
KNN.fit(X_train_knn,y_train)
ypred = KNN.predict(X_test_knn)

Print the classification metric report

In [0]:
accuracy=metrics.accuracy_score(y_test,ypred)
print("The accuracy when is {}".format(accuracy))

In [0]:
n_list = [10,20,30,40,50]

for i in range (0,5):
  KNN = KNeighborsClassifier(n_neighbors= n_list[i])
  KNN.fit(X_train_knn,y_train)
  ypred = KNN.predict(X_test_knn)
  accuracy=metrics.accuracy_score(y_test,ypred)
  print("The accuracy when K equals {} is {}".format(n_list[i], accuracy))

In [0]:
X_train_nn = X_train.reshape((42000, 32, 32, 1))
X_test_nn = X_test.reshape((18000, 32, 32, 1))

Implement and apply a deep neural network classifier including and Batch Normalization for Neural Network

In [0]:
#Initialize model
model = tf.keras.models.Sequential()
#Add first convolutional layer
model.add(tf.keras.layers.Conv2D(32, #Number of filters 
                                 kernel_size=(3,3), #Size of the filter
                                 activation='relu', input_shape=(32,32,1)))
model.add(tf.keras.layers.BatchNormalization())
#Add second convolutional layer
model.add(tf.keras.layers.Conv2D(32, kernel_size=(3,3), activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
#model.output
#Flatten the output
model.add(tf.keras.layers.Flatten())
#Dense layer
model.add(tf.keras.layers.Dense(128, activation='relu'))
#Output layer
model.add(tf.keras.layers.Dense(10, activation='softmax'))
model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [0]:
model.summary()

In [0]:
#Train the model
model.fit(X_train_nn,y_train,          
          validation_data=(X_test_nn,y_test),
          epochs=10,
          batch_size=32)

Understand the differences and trade-offs between traditional and NN classifiers with the help of classification metrics

In [0]:
model.evaluate(X_test_nn,y_test)

In [0]:
# An artificial neural network is a combination of multiple classifiers. In conventional classifiers, 
# there is only one unit that is producing the activation output. But in a Neural network there can be 
# multiple units in a single layer and all of those units individually are classifiers only with some 
# activation function like ReLu, Sigmoid etc. and there can be multiple such layers.

# So one of the main advantage of having such a collection of classifiers is that each of those 
# classifiers can engage themselves for finding different patterns in the data and since there are many
# such units, the network can learn a lot of features from the data.